### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
!pip install -q transformers torch

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openlm-research/open_llama_3b"

# load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

prompt = "Dart vader looked at the skyoker and said: "
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

...

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.85G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Dart vader looked at the skyoker and said: „I am a dart vader. I am a dart vader. I am a dart vader. I am a dart vader. I am a dart vader. I am a dart vader


### Prompt tuning: the story of a fox (2 pts)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))




Output: <s> A quick brown fox jumps over the lazy dog.
A quick brown


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.2949, device='cuda:0', dtype=torch.float16,
       grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
        # This is because we will prepend :num_prompts: padding tokens at the beginning

        # After you are done, you must produce a word embedding vector for each token in input_ids,
        # except that the first :num_prompts: vectors should equal learnable_prompts;
        # any additional vectors after first :num_prompts: ones should be embedded as usual
        # Note: since you're dealing with trainable params, please torch.cat instead of item assignment

        B, L = input_ids.shape
        emb = self.original_word_embeddings(input_ids)
        tr_emb = self.learnable_prompts.expand(B, -1, -1)
        ready_embs = torch.cat([tr_emb, emb[:, self.num_prompts:, :]], dim=1)
        return ready_embs

In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

# space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id, dtype=torch.int64, device=device)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # PAD was None
space_for_prompts = torch.full((test_input_ids.size(0), num_prompts),
                               fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64,
                               device=device)


test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


/tmp/ipython-input-3064492072.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"
prompt_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

del test_emb_layer

In [ ]:

the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
batch_size = batch['input_ids'].size(0)

space_for_prompts = torch.full(
    (batch_size, num_prompts),
    fill_value=tokenizer.pad_token_id,
    dtype=torch.int64,
    device=device
)

# prepend prompts ONCE
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

print(batch['input_ids'], batch['attention_mask'])

opt = torch.optim.Adam([prompt_layer.learnable_prompts], lr=0.01)

for i in range(40):
    opt.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast():
        inputs_embeds = prompt_layer(batch['input_ids'])
        outputs = model(inputs_embeds=inputs_embeds, attention_mask=batch['attention_mask'])
        next_word_logits = outputs.logits[:, num_prompts:-1, :]
        true_next_tokens = batch['input_ids'][:, num_prompts+1:]
        loss = F.cross_entropy(next_word_logits.flatten(0,1), true_next_tokens.flatten(0,1))

    loss.backward()
    opt.step()
    print(f"Step {i}: loss={loss.item()}")

del opt
assert loss.item() <= 0.1
print("Good job!")

tensor([[    2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     1,   308,  2442,  8817,
         24835,   872,   432,  5869,   648,   266, 20793,  3924, 31843, 11973,
         31844,   342,  3924, 21127,   357,  8668, 31905]], device='cuda:0') tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')


/tmp/ipython-input-3138881936.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step 0: loss=4.1737060546875
Step 1: loss=3.56524658203125
Step 2: loss=3.2934327125549316
Step 3: loss=3.08209228515625
Step 4: loss=2.9152464866638184
Step 5: loss=2.7645652294158936
Step 6: loss=2.6121654510498047
Step 7: loss=2.455638885498047
Step 8: loss=2.309349536895752
Step 9: loss=2.170802593231201
Step 10: loss=2.0326879024505615
Step 11: loss=1.9131948947906494
Step 12: loss=1.8064237833023071
Step 13: loss=1.7106361389160156
Step 14: loss=1.6210638284683228
Step 15: loss=1.5164172649383545
Step 16: loss=1.4071859121322632
Step 17: loss=1.2967240810394287
Step 18: loss=1.1834447383880615
Step 19: loss=1.0662063360214233
Step 20: loss=0.9354793429374695
Step 21: loss=0.7814429998397827
Step 22: loss=0.5983463525772095
Step 23: loss=0.43592244386672974
Step 24: loss=0.33833199739456177
Step 25: loss=0.2682456970214844
Step 26: loss=0.1973838359117508
Step 27: loss=0.14786288142204285
Step 28: loss=0.12307904660701752
Step 29: loss=0.10626830905675888
Step 30: loss=0.092372953

In [ ]:
prompt = 'A quick brown fox'

batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

model.eval()

for _ in range(20):
    with torch.no_grad(), torch.cuda.amp.autocast():
        inputs_embeds = prompt_layer(batch['input_ids'])

        outputs = model(
            inputs_embeds=inputs_embeds,
            attention_mask=batch['attention_mask']
        )

        next_token = outputs.logits[:, -1].argmax(-1, keepdim=True)

    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=1)
    batch['attention_mask'] = torch.cat(
        [batch['attention_mask'], torch.ones_like(next_token)], dim=1
    )


print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))
# original code did not included trained mebeddings some bug I do not know.

# if you did everything right, the model will deny that the fox jumped over the lazy dog

/tmp/ipython-input-3323588620.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():



Output: <s> A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway! Through the mischie


### Using HuggingFace PEFT (2 points)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [ ]:

del inputs
del outputs

In [ ]:




import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 51200
Total parameters (excluding quantization): 3426524800


In [ ]:
NUM_VIRTUAL_TOKENS = 16
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

truth_sentence = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(truth_sentence, return_tensors="pt").to(device)
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

for step in range(30):
    opt.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    logits = outputs.logits[:, NUM_VIRTUAL_TOKENS:-1, :]
    labels = input_ids[:, NUM_VIRTUAL_TOKENS+1:]

    # handle short sequences
    min_len = min(logits.shape[1], labels.shape[1])
    logits = logits[:, :min_len, :]
    labels = labels[:, :min_len]

    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
    loss.backward()
    opt.step()

    print(f"Step {step+1}/30 | Loss: {loss.item():.4f}")




Step 1/30 | Loss: 12.0078
Step 2/30 | Loss: 10.5391
Step 3/30 | Loss: 9.2266
Step 4/30 | Loss: 8.4062
Step 5/30 | Loss: 7.6406
Step 6/30 | Loss: 6.8047
Step 7/30 | Loss: 5.9570
Step 8/30 | Loss: 5.1328
Step 9/30 | Loss: 4.3789
Step 10/30 | Loss: 3.7070
Step 11/30 | Loss: 3.1523
Step 12/30 | Loss: 2.7480
Step 13/30 | Loss: 2.4199
Step 14/30 | Loss: 2.1445
Step 15/30 | Loss: 1.8721
Step 16/30 | Loss: 1.5908
Step 17/30 | Loss: 1.2793
Step 18/30 | Loss: 0.9551
Step 19/30 | Loss: 0.6172
Step 20/30 | Loss: 0.2883
Step 21/30 | Loss: 0.0859
Step 22/30 | Loss: 0.0350
Step 23/30 | Loss: 0.0229
Step 24/30 | Loss: 0.0187
Step 25/30 | Loss: 0.0165
Step 26/30 | Loss: 0.0146
Step 27/30 | Loss: 0.0136
Step 28/30 | Loss: 0.0129
Step 29/30 | Loss: 0.0123
Step 30/30 | Loss: 0.0112


In [ ]:
# Feel free to structure your code as you see fit - as long as it's legible :)

def get_outputs(model, inputs, max_new_tokens=20):
    with torch.no_grad():
      outputs = model.generate(
          input_ids=inputs["input_ids"],
          attention_mask=inputs["attention_mask"],
          max_new_tokens=max_new_tokens,
      )
    return outputs

prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

# Just pass batch as is
loaded_model_prompt_outputs = get_outputs(model, batch)
print(tokenizer.batch_decode(loaded_model_prompt_outputs, skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


['A quick brown fox anyway\nAnd anyway\n$$. Anyway, anyway, anyway\n$$.\n$$.\n$']


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [ ]:

del model
del opt
del input_ids
del outputs
del prompt_layer

import gc
gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   6608 MiB |  13599 MiB | 348946 MiB | 342338 MiB |
|       from large pool |   6606 MiB |  13387 MiB | 268857 MiB | 262251 MiB |
|       from small pool |      2 MiB |    306 MiB |  80089 MiB |  80086 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   6608 MiB |  13599 MiB | 348946 MiB | 342338 MiB |
|       from large pool |   6606 MiB |  13387 MiB | 268857 MiB |

In [ ]:
# re-load the model to remove any previous PEFT tuners


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

        self.adapter_A.requires_grad=True
        self.adapter_B.requires_grad=True

        # self.scaling = rank ** -0.5

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        #  <YOUR CODE HERE>
        model_out = self.module(input)
        lora_out =(input @ self.adapter_A @ self.adapter_B)
        return model_out + lora_out

In [ ]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [ ]:
lora_rank = 8

print(model)

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device) # <- not sure it is good idea, usually only Q, V is made (redundant)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

print(sum(isinstance(module, LoRALayer) for module in model.modules()))
# assert sum(isinstance(module, LoRALayer) for module in model.modules()) ==   52# for Llama-7B, not right as I changed model to LLAMA-3b it is 78

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 3200, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (k_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (v_proj): Linear(in_features=3200, out_features=3200, bias=False)
          (o_proj): Linear(in_features=3200, out_features=3200, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (up_proj): Linear(in_features=3200, out_features=8640, bias=False)
          (down_proj): Linear(in_features=8640, out_features=3200, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((3200,), ep

In [ ]:
!nvidia-smi

Wed Jan 21 13:24:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             28W /   70W |   13636MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
torch.cuda.empty_cache()
del batch
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)

# test a single training step, make sure we get meaningful gradients

with torch.cuda.amp.autocast(dtype=torch.float32):
      out = model.forward(**batch)
      (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

## run out of memory to show gradients :(
## but trainign below works and loss lowers over time! which implies gradients are ok

/tmp/ipython-input-1624731278.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [ ]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs', report_to=None),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
1,1.011200
2,0.321700
3,1.293400
4,1.045300
5,0.713100
6,1.392200
7,1.529900
8,1.107100
9,0.576300


OutOfMemoryError: CUDA out of memory. Tried to allocate 68.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 15154 has 14.73 GiB memory in use. Of the allocated memory 14.28 GiB is allocated by PyTorch, and 316.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Final task: *actually* train the model (4 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [ ]:
# del model


import gc
gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 2            |        cudaMalloc retries: 3         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  14691 MiB |  14691 MiB | 384159 MiB | 369467 MiB |
|       from large pool |  14658 MiB |  14658 MiB | 294185 MiB | 279526 MiB |
|       from small pool |     33 MiB |    306 MiB |  89974 MiB |  89940 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  14691 MiB |  14691 MiB | 384159 MiB | 369467 MiB |
|       from large pool |  14658 MiB |  14658 MiB | 294185 MiB |

In [2]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

from transformers import AutoTokenizer
model_id = "openlm-research/open_llama_3b"

tokenizer = AutoTokenizer.from_pretrained(model_id)

from datasets import load_dataset
dataset = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)

new_data = []

for _ in range(1000):
  new_data.append(next(iter(dataset)))
from datasets import Dataset
print(new_data[0])
hf_dataset = Dataset.from_list(new_data)
hf_dataset = hf_dataset.map(lambda samples: {"content": samples["content"][:512]})

data = hf_dataset.map(lambda samples: tokenizer(samples['content']), batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/593 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/534k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggin

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

{'repo_name': 'ahmedbodi/AutobahnPython', 'path': 'examples/asyncio/websocket/echo/client_coroutines.py', 'copies': '13', 'size': '2044', 'content': '###############################################################################\n##\n##  Copyright (C) 2013-2014 Tavendo GmbH\n##\n##  Licensed under the Apache License, Version 2.0 (the "License");\n##  you may not use this file except in compliance with the License.\n##  You may obtain a copy of the License at\n##\n##      http://www.apache.org/licenses/LICENSE-2.0\n##\n##  Unless required by applicable law or agreed to in writing, software\n##  distributed under the License is distributed on an "AS IS" BASIS,\n##  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n##  See the License for the specific language governing permissions and\n##  limitations under the License.\n##\n###############################################################################\n\nfrom autobahn.asyncio.websocket import WebSocketClientPro

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
class LoRALayer(nn.Module):
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # frozen pre-trained layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5**0.5)

    def forward(self, x):
        original_out = self.module(x)
        lora_out = x @ self.adapter_A @ self.adapter_B
        return original_out + lora_out

## added scaling (stability +)

In [7]:
del model
# del batch
del optimizer

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "openlm-research/open_llama_3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")
for param in model.parameters():
    param.requires_grad=False #<- to train LoRa only!
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_rank = 8

for layer in model.model.layers:
    layer.self_attn.q_proj = LoRALayer(layer.self_attn.q_proj, rank=lora_rank).to(device)
    layer.self_attn.v_proj = LoRALayer(layer.self_attn.v_proj, rank=lora_rank).to(device)

model.lm_head = LoRALayer(model.lm_head, rank=lora_rank).to(device)

print(f"Total LoRA layers: {sum(isinstance(m, LoRALayer) for m in model.modules())}")

print(sum(isinstance(module, LoRALayer) for module in model.modules()))

ModuleList(
  (0-25): 26 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=3200, out_features=3200, bias=False)
      (k_proj): Linear(in_features=3200, out_features=3200, bias=False)
      (v_proj): Linear(in_features=3200, out_features=3200, bias=False)
      (o_proj): Linear(in_features=3200, out_features=3200, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=3200, out_features=8640, bias=False)
      (up_proj): Linear(in_features=3200, out_features=8640, bias=False)
      (down_proj): Linear(in_features=8640, out_features=3200, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
    (post_attention_layernorm): LlamaRMSNorm((3200,), eps=1e-06)
  )
)
LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=3200, out_features=3200, bias=False)
    (k_proj): Linear(in_features=3200, out_features=3200, bias=False)
    (v_proj): Linear(in_fea

In [21]:

def tokenize_function(examples):
    return tokenizer(
        examples["content"],
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

tokenized_ds = hf_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=hf_dataset.column_names
)

from torch.utils.data import DataLoader
from tqdm import tqdm

class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data["input_ids"]
        self.attention_mask = tokenized_data["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(self.input_ids[idx], dtype=torch.long)
        }

train_dataset = CodeDataset(tokenized_ds)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=0.01
)

model.train()
num_epochs = 2

for epoch in range(num_epochs):
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        pbar.set_postfix({'loss': loss.item()})

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss / len(train_loader):.4f}")

    ## was trained on 3 epoch (rerun 2 times) 1000 samples in each epoch, and genartes python code but can sometimes start genartion on another programming languages

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch 1: 100%|██████████| 250/250 [02:53<00:00,  1.44it/s, loss=7.12e-6]


Epoch 1 | Avg Loss: 0.0000


Epoch 2: 100%|██████████| 250/250 [02:52<00:00,  1.45it/s, loss=5.9e-6]

Epoch 2 | Avg Loss: 0.0000


In [22]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms
import pandas as pd
def generate_text(model, prompt, max_new_tokens=50):
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=False,
        truncation=False
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            early_stopping=True
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if full_text.startswith(prompt):
        gen_text = full_text[len(prompt):]
    else:
        gen_text = full_text

    gen_text = gen_text.split(tokenizer.eos_token)[0].strip()
    return gen_text


prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch', 'fast car', "apple"]

print("Generating baseline samples...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16
).to(device)
baseline_model.eval()

before_samples = [generate_text(baseline_model, p) for p in prompts]
after_samples = [generate_text(model, p) for p in prompts]

del baseline_model
torch.cuda.empty_cache()

df = pd.DataFrame({
    "Prompt": [f"`{p}`" if p else "`<empty>`" for p in prompts],
    "Before Fine-tuning": before_samples,
    "After Fine-tuning": after_samples
})

df = df.replace({r'\n': '<br>', r'  ': '&nbsp;&nbsp;'}, regex=True)

html_str = df.to_html(escape=False, index=False, table_id="code-table")
html_str = html_str.replace('<table', '<table style="border-collapse: collapse; width: 100%;"')
html_str = html_str.replace('<th>', '<th style="border: 1px solid #aaa; padding: 8px; background-color: #f2f2f2;">')
html_str = html_str.replace('<td>', '<td style="border: 1px solid #ddd; padding: 6px; vertical-align: top; font-family: monospace;">')

display(HTML(html_str))

# nit realated to programming start tokens will continue in its ows style not python code

Generating baseline samples...


Prompt,Before Fine-tuning,After Fine-tuning
``,The latest version of our popular software is available now.We have been developing a new version of our popular software for a while now. I have been working with the developers behind this software to make sure that the new version will be even better than,################################################################################### Copyright (C) 2013-2014 Tavendo GmbH#### Licensed
`import`,"osimport sysif __name__ == ""__main__"": os.environ.setdefault(""DJANGO_SETTINGS_MODULE"", ""troop_bot.settings"") from",{ NgModule } from '@angular/core';import { BrowserModule } from '@angular/platform-browser';import { HttpModule } from '@angular/http';
`from`,__future__ import absolute_importimport osimport sysfrom oslo.config import cfgimport eventletfrom nova import exceptionfrom nova.i18n import _from,"django.contrib.auth.models import Userfrom django.contrib.auth.models import AnonymousUserclass UserManager(BaseUserManager): def __init__(self, *args, **kwar"
`while`,(true) { var n = 20000; var t = 10000; var x = 1; if (x > n - t) x = n - t;,"i was walking down the street with my wife i saw a little old lady with a stick.She walked slowly, but she walked.She was the sweetest thing i had ever seen.when she finally came to where i stood,"
`try`,{ $_SESSION['url'] = $_SERVER['REQUEST_URI'];} catch (Exception $ex) { $_SESSION['url'] = '';}print_,{\t\tvar _ = require('../util/internal/_');\t\tmodule.exports = _.isObject = function (obj) {\t\t var type = typeof obj;\t\t return type ===
`if`,"it was not the case, he had a great time.So, I told him that I did not understand.I told him that I did not understand at all.He said that the same would be the case.It would","[ ! -d ""../lib"" ]; then mkdir -p ""../lib""fi###### BEGIN LICENSE###### Copyright (C) 20"
`for`,"the first time in its history, theThe 100-year-old company has been able to achieve these results thanks to the digital transformation that began in 2016 with the implementation of Microsoft Dynamics 365 Business",those who have the time to read all of this:http://www.cnn.com/2009/POLITICS/10/24/tapper.tapper.cnn.com/I
`torch`,"bearers,The National Museum of the United States Navy is open to the public from 10 a.m. until 4:30 p.m. daily and admission is free. The museum is located in the Navy Yard area",.initialize(torch);################################################################################### Copyright (C) 2018-2019 T
`fast car`,"racing games free downloadGaming is something that can be enjoyed by anyone. There are many different kinds of games, and most of them are very addictive. The best part about gaming is that you can play it on any platform. Gaming is","50 Best Driving Songs of All Time, Ranked by Popularity, Ranked (Part 2)By Jeff Tain 2 CommentsTags: 50 Best Driving Songs of All Time, Alan Jackson, Alanis Mor"
`apple`,"_newsFollowing an unprecedented 2020, 2021 saw a record number of new vehicles sold in the U.S.Sales of new cars and trucks reached their highest level since 2001","banana beetleApple-boring beetle10.4 – 14 mmIn the United States, the apple-boring beetle is found only in the Pacific Northwest.It is found from southwestern British"


If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.